## RFM analüüs UrbanStyle.ltd jaoks
Turundusanalüüsi osakond  
15.05.2026  
Marko väljakutse

### Roll A - Data Loading

In [ ]:
import os
from dotenv import load_dotenv
from supabase import create_client
import pandas as pd
import plotly.express as px



load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)


try:
    response = supabase.table("sales").select("*").limit(1).execute()

    print("Ühendus Supabase'iga töötab!")
    print(response.data)

except Exception as e:
    print("Viga ühendamisel:")
    print(e)


#sales tabel
all_sales = []
page_size = 1000
start = 0

while True:
    response = (
        supabase
        .table("sales")
        .select("*")
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data

    if not data:
        break

    all_sales.extend(data)

    print(f"Laetud: {len(all_sales)} rida")

    start += page_size

df_sales = pd.DataFrame(all_sales)

print(df_sales.shape)
df_sales.head()


#customer tabel
all_customers = []
page_size = 1000
start = 0

while True:
    response = (
        supabase
        .table("customers")
        .select("*")
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data

    if not data:
        break

    all_customers.extend(data)

    print(f"Laetud: {len(all_customers)} rida")

    start += page_size

df_customers = pd.DataFrame(all_customers)

print(df_customers.shape)
df_customers.head()

#ühendan
df = pd.merge(
    df_sales,
    df_customers,
    on="customer_id",
    how="left"
)

# Muudame tüübid ära
df["customer_id"] = df["customer_id"].astype("Int64")
df["birth_year"] = df["birth_year"].astype("Int64")

#kontroll
print(df.shape)
print(df.dtypes)

df.head()

#email kontroll
print(df[["customer_id", "sale_date", "total_price", "email"]].head())

print("Puuduvaid email'e:", df["email"].isna().sum())

#lõpp
print("Sales read:", df_sales.shape)
print("Customers read:", df_customers.shape)
print("Merged df:", df.shape)

# lisa kontrollraport
print("Ridade arv:", df.shape[0])
print("Veergude arv:", df.shape[1])
print("Unikaalseid kliente:", df["customer_id"].nunique())
print("Müük kokku:", df["total_price"].sum())
print("Puuduvaid email'e:", df["email"].isna().sum())

# salvesta csv
df.to_csv("merged_sales_customers.csv", index=False)

print("Fail salvestatud: merged_sales_customers.csv")

#kus fail salvestub
import os

print(os.getcwd())


### Roll B — Data Cleaning

In [ ]:
# Roll B algus — loen Roll A poolt salvestatud CSV sisse. Tööversioon df_clean

import pandas as pd

df_clean = pd.read_csv("merged_sales_customers.csv")

print("Roll B algne shape:", df_clean.shape)
df_clean.head()

In [ ]:
# 1.A Duplikaatide kontroll ja eemaldamine

duplicates_before = df_clean.duplicated().sum()

print("Duplikaadid enne eemaldamist:", duplicates_before)
print("Shape enne duplikaatide eemaldamist:", df_clean.shape)

In [ ]:
# 1B. Duplikaatide eemaldamine

df_clean = df_clean.drop_duplicates()

duplicates_after = df_clean.duplicated().sum()

print("Duplikaadid pärast eemaldamist:", duplicates_after)
print("Shape pärast duplikaatide eemaldamist:", df_clean.shape)

In [ ]:
# 2A. NULL väärtuste kontroll - 

print("NULL-id veergude kaupa:")
print(df_clean.isnull().sum())

In [ ]:
# 2B. Kriitiliste NULL väärtustega ridade eemaldamine

rows_before_null_cleaning = df_clean.shape[0]

df_clean = df_clean.dropna(subset=['customer_id', 'sale_date', 'total_price'])

rows_after_null_cleaning = df_clean.shape[0]

print("Ridu enne kriitiliste NULL-ide eemaldamist:", rows_before_null_cleaning)
print("Ridu pärast kriitiliste NULL-ide eemaldamist:", rows_after_null_cleaning)
print("Eemaldatud ridu:", rows_before_null_cleaning - rows_after_null_cleaning)

print("\nKriitiliste veergude NULL-id pärast puhastamist:")
print(df_clean[['customer_id', 'sale_date', 'total_price']].isnull().sum())

In [ ]:
# 3. sale_date teisendamine datetime formaati — enne ja pärast

print("=== ENNE TEISENDAMIST ===")
print("sale_date andmetüüp enne:", df_clean['sale_date'].dtype)
print(df_clean['sale_date'].head())

df_clean['sale_date'] = pd.to_datetime(df_clean['sale_date'])

print("\n=== PÄRAST TEISENDAMIST ===")
print("sale_date andmetüüp pärast:", df_clean['sale_date'].dtype)
print(df_clean['sale_date'].head())
print("Kuupäevavahemik:", df_clean['sale_date'].min(), "kuni", df_clean['sale_date'].max())

In [ ]:
# 4A. total_price väärtuste kontroll

print("Ridade arv enne total_price puhastamist:", df_clean.shape[0])
print("Negatiivsed total_price väärtused:", (df_clean['total_price'] < 0).sum())
print("Null total_price väärtused:", (df_clean['total_price'] == 0).sum())
print("Null või negatiivsed total_price väärtused:", (df_clean['total_price'] <= 0).sum())

print("total_price miinimum:", df_clean['total_price'].min())
print("total_price maksimum:", df_clean['total_price'].max())

In [ ]:
# 4B. Null või negatiivse total_price väärtusega ridade eemaldamine

rows_before_price_cleaning = df_clean.shape[0]

df_clean = df_clean[df_clean['total_price'] > 0]

rows_after_price_cleaning = df_clean.shape[0]

print("Ridu enne total_price puhastamist:", rows_before_price_cleaning)
print("Ridu pärast total_price puhastamist:", rows_after_price_cleaning)
print("Eemaldatud ridu:", rows_before_price_cleaning - rows_after_price_cleaning)

print("\nKontroll pärast puhastamist:")
print("Null või negatiivsed total_price väärtused:", (df_clean['total_price'] <= 0).sum())
print("total_price miinimum pärast puhastamist:", df_clean['total_price'].min())

In [ ]:
# 5. PUHASTUSRAPORT:

# 5. Puhastusraport

print("=== PUHASTUSRAPORT ===")

print("Lõplik shape:", df_clean.shape)
print("Unikaalseid kliente:", df_clean['customer_id'].nunique())
print("Kuupäevavahemik:", df_clean['sale_date'].min(), "kuni", df_clean['sale_date'].max())

print("\nDuplikaadid lõplikus andmestikus:", df_clean.duplicated().sum())

print("\nKriitiliste veergude NULL-id:")
print(df_clean[['customer_id', 'sale_date', 'total_price']].isnull().sum())

print("\ntotal_price kontroll:")
print("Miinimum:", df_clean['total_price'].min())
print("Maksimum:", df_clean['total_price'].max())

df_clean.head()

### Roll C - RFM analysis

In [ ]:
# My role: Role C - RFM analysis.

# reference_date corresponds to "today" in the teamwork manual.
reference_date = pd.to_datetime("2025-02-28")

df_for_rfm = df_clean[df_clean["sale_date"] <= reference_date]

rfm = df_for_rfm.groupby("customer_id").agg(
    last_purchase=("sale_date", "max"),
    total_purchases=("id", "size"),
    total_spending=("total_price", "sum")
).reset_index()

rfm["days_since_last_purchase"] = (reference_date - rfm["last_purchase"]).dt.days

q = 5 # number of quantiles
ratings = list(range(1, q + 1)) # for F and M scores
reversed_ratings = list(range(q, 0, -1)) # for R score - the smaller the days_since_last_purchase value, the higher the rating
print("Possible R ratings:", reversed_ratings)
print("Possible F and M ratings:", ratings)

# R score (recency).
rfm["R_score"] = pd.qcut(rfm["days_since_last_purchase"].rank(method="first"), q, labels=reversed_ratings).astype(int)

# F score (frequency).
# Encountered an error: ValueError: Bin edges must be unique: Index([1.0, 2.0, 2.0, 3.0, 5.0, 77.0], dtype='float64', name='total_purchases').
# AI suggested to use .rank(method="first") to fix the problem.
rfm["F_score"] = pd.qcut(rfm["total_purchases"].rank(method="first"), q, labels=ratings).astype(int)

# M score (monetary).
rfm["M_score"] = pd.qcut(rfm["total_spending"].rank(method="first"), q, labels=ratings).astype(int)

# RFM score as sum of individual scores.
rfm["RFM_score"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

# Segmentation.
# 13-15 = VIP Champions, 10-12 = Loyal, 7-9 = Potential, 4-6 = At Risk, 3 = Lost.
# Return numeric values to allow meaningful sorting.
def to_segment_value(score):
    if score >= 13:
        return 4
    elif score >= 10:
        return 3
    elif score >= 7:
        return 2
    elif score >= 4:
        return 1
    else:
        return 0

segment_labels = ("Lost", "At Risk", "Potential", "Loyal", "VIP Champions")

rfm["segment_value"] = rfm["RFM_score"].apply(to_segment_value)
rfm["segment"] = rfm["segment_value"].apply(lambda x: segment_labels[x])

segments = rfm.groupby(["segment", "segment_value"]).agg(
    total_customers=("segment_value", "size"),
    total_purchases=("total_purchases", "sum"),
    total_spending=("total_spending", "sum"),
    average_spending=("total_spending", "mean")
).sort_values("segment_value", ascending=False).reset_index()

segments["total_spending_share"] = segments["total_spending"] / segments["total_spending"].sum()

segments # This will pretty-print the result in Jupyter notebook.

### Roll D - Visualization

In [ ]:
# Impordin Plotly
import plotly.express as px

# Loon tulpdiagrammi, mis näitab segmentide klientide arvu
fig1 = px.bar(
    segments,
    x='segment',
    y='total_customers',
    title='UrbanStyle kliendibaasi jaotus segmentide lõikes',
    labels={'segment': 'Kliendisegment', 'total_customers': 'Klientide arv'},
    text='total_customers',
    color_discrete_sequence=['#009B8D']
)

# Viimistlen diagrammi - eemaldan x-telje pealkirja, muudan y-telje pealkirja, muudan tausta valgeks
fig1.update_layout(
    xaxis_title="", 
    yaxis_title="Klientide arv",
    plot_bgcolor='rgba(0,0,0,0)'
)

fig1.show()

In [ ]:
# Loon RFM hajuvusdiagrammi, mis näitab kliendisuhete dünaamikat: Recency vs Monetary
fig2 = px.scatter(
    rfm,
    x='days_since_last_purchase', 
    y='total_spending', 
    color='segment', 
    size='total_purchases',
    title='UrbanStyle RFM-analüüs: Recency vs Monetary',
    labels={
        'days_since_last_purchase': 'Päevi viimasest ostust', 
        'total_spending': 'Kogukulutus (€)',
        'segment': 'Segment',
        'total_purchases': 'Ostude arv'
    },
    hover_data=['customer_id']
)

fig2.show()

In [ ]:
# Loon tulpdiagrammi, mis näitab TOP 10 VIP klienti kogukulutuse järgi
top_10_vips = rfm[rfm['segment'] == 'VIP Champions'].nlargest(10, 'total_spending')

# Muudan kliendi id väärtuse esmalt täisarvuks ja siis tekstiks, et Plotly ei prooviks seda graafikul uuesti numbrilise skaalana tõlgendada, vaid käsitleks seda diskreetse sildina
top_10_vips['customer_id'] = top_10_vips['customer_id'].astype(int).astype(str)
top_10_vips['total_spending'] = top_10_vips['total_spending'].round()

fig3 = px.bar(
    top_10_vips,
    x='total_spending',
    y='customer_id', # Võid asendada nimega, kui oled customers tabeliga liitnud
    orientation='h',
    title='UrbanStyle TOP 10 VIP klienti kogukulutuse järgi',
    labels={'total_spending': 'Kogukulutus (€)', 'customer_id': 'Kliendi ID'},
    color_discrete_sequence=['#009B8D'],
    text='total_spending'
)

# Järjestan tulbad suuruse järgi, teen tausta valgeks
fig3.update_layout(
    yaxis={'categoryorder':'total ascending'},
    plot_bgcolor='rgba(0,0,0,0)'
)

fig3.show()

In [ ]:
# Leian kõikide segmentide osakaalu kogukäibest
# 1. Grupeerin segmendi järgi ja liidan kokku nende kulutused
segmentide_analuus = segments.groupby('segment')['total_spending'].sum().reset_index()

# 2. Leian terve ettevõtte kogukäibe (kõikide segmentide summa)
ettevotte_kogukäive = segmentide_analuus['total_spending'].sum()

# 3. Arvutan iga segmendi protsendilise osakaalu
segmentide_analuus['osakaal_%'] = (segmentide_analuus['total_spending'] / ettevotte_kogukäive) * 100

# 4. Sorteerin tulemuse kahanevalt, et kõige väärtuslikumad grupid oleksid eespool
segmentide_analuus = segmentide_analuus.sort_values(by='osakaal_%', ascending=False)

# Kuvan tulemuse ümardatuna (nii on mugavam lugeda)
print("UrbanStyle'i segmentide käibeosakaalud:")
print(segmentide_analuus[['segment', 'total_spending', 'osakaal_%']].round(1))

### Äritõlgendus:
1. ***VIP Champions* (453 klienti):** Tuvastatud on 453 tippklienti, kes on UrbanStyle'i äri kõige olulisemad kliendid. Just see väike, kuid lojaalne grupp annab ligi 43% meie kogukäibest. Nad ostavad sagedasti, kulutavad keskmisest oluliselt rohkem ja on teinud oma viimase ostu hiljuti.  

2. ***At-Risk* kliendid (525 klienti):** Tuvastasime 525 "At-Risk" klienti, mis on kriitiline ohumärk. Tegemist on klientidega, kes olid varem väga väärtuslikud ja sagedased ostjad, kuid pole nüüdseks juba kuude kaupa ühtegi tehingut teinud. Kuna see segment on arvuliselt suurem kui meie VIP-ide rühm, on potentsiaalne käibekadu märkimisväärne, kui me kohe ei sekku.  

3. ***Potential* kliendid (768 klienti):** See on meie suurim segment ja ühtlasi suurim kasvuvõimalus. Need kliendid on hiljuti ostnud ja näitavad positiivset huvi, kuid nad vajavad täiendavat "tõuget", et muutuda püsiklientideks.

<br>

### Soovitused:
Kuna segmenteeritud kampaaniad on kuni kolm korda efektiivsemad kui üldised pakkumised, soovitame turundusjuhile järgmisi samme:
* **VIP programm (eritingimused *VIP Champions* segmendile):**
   - Fookus peab olema eksklusiivsusel ja tunnustusel, mitte massilistel allahindlustel.
   - Pakume neile varajast ligipääsu uutele kollektsioonidele, personaalseid tänusõnumeid ja alati tasuta tarnet, et premeerida nende lojaalsust ilma marginaali liigselt kahjustamata.
* **Win-back kampaania (*At-Risk* klientidele):**
   - Eesmärk on võita nad tagasi enne, kui nad muutuvad lõplikult "kadunud" klientideks.
   - Saadame neile personaalse e-kirja sõnumiga "Me igatseme teid!", mis sisaldab tootesoovitusi nende varasema ostuajaloo põhjal ja ajaliselt piiratud (7 päeva) 15% sooduskoodi, et tekitada kiireloomulisust.
* **Uute klientide kaasamine (*Nurture* programm *Potential* segmendile):**
   - Kasutame lojaalsuspunktide süsteemi, kus iga ost viib kliendi järgmisele tasemele.
   - Rakendame suunatud sõnumeid nagu "Veel üks ost ja sa oled VIP!", et julgustada neid looma UrbanStyle'ist ostmise harjumust ja tõsta nende ostusagedust.

Kirjeldatud andmepõhine lähenemine asendab senise "kõhutunde" strateegilise juhtimisega, võimaldades meil optimeerida turunduseelarvet ja maksimeerida klientide lojaalsust ja ostusagedust.

<br>

### Puuduvad andmed:
* Tagastatud tellimuste info